# Classification of Pet's Real-Life Images

Lab Assignment from [AI for Beginners Curriculum](https://github.com/microsoft/ai-for-beginners).

Now it's time to deal with more challenging task - classification of the original [Oxford-IIIT Dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/). Let's start by loading and visualizing the dataset.

In [ ]:
# !wget https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
# !tar xfz images.tar.gz
# !rm images.tar.gz

We will define generic function to display a series of images from a list:

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np

def display_images(l,titles=None,fontsize=12):
    n=len(l)
    fig,ax = plt.subplots(1,n)
    for i,im in enumerate(l):
        ax[i].imshow(im)
        ax[i].axis('off')
        if titles is not None:
            ax[i].set_title(titles[i],fontsize=fontsize)
    fig.set_size_inches(fig.get_size_inches()*n)
    plt.tight_layout()
    plt.show()

You can see that all images are located in one directory called `images`, and their name contains the name of the class (breed):

In [ ]:
fnames = os.listdir('images')[:5]
display_images([Image.open(os.path.join('images',x)) for x in fnames],titles=fnames,fontsize=30)

To simplify classification and use the same approach to loading images as in the previous part, let's sort all images into corresponding directories:

In [ ]:
for fn in os.listdir('images'):
    cls = fn[:fn.rfind('_')].lower()
    os.makedirs(os.path.join('images',cls),exist_ok=True)
    os.replace(os.path.join('images',fn),os.path.join('images',cls,fn))

Let's also define the number of classes in our dataset:

In [ ]:
num_classes = len(os.listdir('images'))
num_classes

## Preparing dataset for Deep Learning

To start training our neural network, we need to convert all images to tensors, and also create tensors corresponding to labels (class numbers). Most neural network frameworks contain simple tools for dealing with images:
* In Tensorflow, use `tf.keras.preprocessing.image_dataset_from_directory`
* In PyTorch, use `torchvision.datasets.ImageFolder`

As you have seen from the pictures above, all of them are close to square image ratio, so we need to resize all images to square size. Also, we can organize images in minibatches.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchinfo import summary

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Define transforms - resize to 224x224 for CNN input
trans = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load dataset using ImageFolder
dataset = torchvision.datasets.ImageFolder('images', transform=trans)
print(f'Dataset size: {len(dataset)} images')
print(f'Number of classes: {len(dataset.classes)}')

Now we need to separate dataset into train and test portions:

In [ ]:
# Split dataset into train and test sets (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

trainset, testset = torch.utils.data.random_split(
    dataset, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # for reproducibility
)

print(f'Training set size: {len(trainset)}')
print(f'Test set size: {len(testset)}')

Now define data loaders:

In [ ]:
# Define data loaders
batch_size = 32

train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False)

print(f'Number of training batches: {len(train_loader)}')
print(f'Number of test batches: {len(test_loader)}')

In [ ]:
# [OPTIONAL] Plot some samples from the dataset
def display_dataset(dataset, n=10):
    import numpy as np
    fig, ax = plt.subplots(1, n, figsize=(15, 3))
    for i in range(n):
        img, label = dataset[i]
        img_np = img.permute(1, 2, 0).numpy()
        ax[i].imshow(img_np)
        ax[i].axis('off')
        ax[i].set_title(dataset.dataset.classes[label], fontsize=8)
    plt.tight_layout()
    plt.show()

display_dataset(trainset, n=8)

## Define a neural network

For image classification, you should probably define a convolutional neural network with several layers. What to keep an eye for:
* Keep in mind the pyramid architecture, i.e. number of filters should increase as you go deeper
* Do not forget activation functions between layers (ReLU) and Max Pooling
* Final classifier can be with or without hidden layers, but the number of output neurons should be equal to number of classes.

An important thing is to get the activation function on the last layer + loss function right:
* In Tensorflow, you can use `softmax` as the activation, and `sparse_categorical_crossentropy` as loss. The difference between sparse categorical cross-entropy and non-sparse one is that the former expects output as the number of class, and not as one-hot vector.
* In PyTorch, you can have the final layer without activation function, and use `CrossEntropyLoss` loss function. This function applies softmax automatically. 

> **Hint:** In PyTorch, you can use `LazyLinear` layer instead of `Linear`, in order to avoid computing the number of inputs. It only requires one `n_out` parameter, which is number of neurons in the layer, and the dimension of input data is picked up automatically upon first `forward` pass.

In [ ]:
# Define a simple CNN for image classification
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 224 -> 112
            
            # Conv Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 112 -> 56
            
            # Conv Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 56 -> 28
            
            # Conv Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 28 -> 14
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model
model = SimpleCNN(num_classes=num_classes).to(device)
print(summary(model, (1, 3, 224, 224)))

## Train the Neural Network

Now we are ready to train the neural network. During training, please collect accuracy on train and test data on each epoch, and then plot the accuracy to see if there is overfitting.

In [ ]:
# Training function
def train_epoch(model, dataloader, optimizer, loss_fn):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(dataloader), correct / total

def validate(model, dataloader, loss_fn):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(dataloader), correct / total

# Train the model
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
epochs = 5

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn)
    val_loss, val_acc = validate(model, test_loader, loss_fn)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1}/{epochs}: '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

In [ ]:
# Plot training results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy plot
ax1.plot(history['train_acc'], label='Train Accuracy')
ax1.plot(history['val_acc'], label='Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training and Validation Accuracy')
ax1.legend()
ax1.grid(True)

# Loss plot
ax2.plot(history['train_loss'], label='Train Loss')
ax2.plot(history['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Training and Validation Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f'Final Validation Accuracy: {history["val_acc"][-1]*100:.2f}%')

Even if you have done everything correctly, you will probably see that the accuracy is quite low.

## Transfer Learning

To improve the accuracy, let's use pre-trained neural network as feature extractor. Feel free to experiment with VGG-16/VGG-19 models, ResNet50, etc.

> Since this training is slower, you may start with training the model for the small number of epochs, eg. 3. You can always resume training to further improve accuracy if needed.

We need to normalize our data differently for transfer learning, thus we will reload the dataset again using different set of transforms:

In [ ]:
# LOAD THE DATASET with proper normalization for pre-trained models
# Standard normalization for ImageNet pre-trained models (VGG, ResNet, etc.)
std_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

trans_vgg = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    std_normalize
])

# Reload dataset with VGG transforms
dataset_vgg = torchvision.datasets.ImageFolder('images', transform=trans_vgg)

# Split into train and test
train_size = int(0.8 * len(dataset_vgg))
test_size = len(dataset_vgg) - train_size
trainset_vgg, testset_vgg = torch.utils.data.random_split(
    dataset_vgg, [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader_vgg = torch.utils.data.DataLoader(trainset_vgg, batch_size=16, shuffle=True)
test_loader_vgg = torch.utils.data.DataLoader(testset_vgg, batch_size=16, shuffle=False)

print(f'Dataset loaded with VGG normalization')
print(f'Training samples: {len(trainset_vgg)}, Test samples: {len(testset_vgg)}')

Let's load the pre-trained network:

In [ ]:
# Load pre-trained VGG16 model
vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
print(vgg)

Now define the classification model for your problem:
* In PyTorch, there is a slot called `classifier`, which you can replace with your own classifier for the desired number of classes.
* In TensorFlow, use VGG network as feature extractor, and build a `Sequential` model with VGG as first layer, and your own classifier on top

In [ ]:
# BUILD MODEL for your problem with your own linear layers
# Replace the classifier with a custom one for our number of classes
vgg.classifier = nn.Sequential(
    nn.Linear(512 * 7 * 7, 4096),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(4096, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, num_classes)
)

model_vgg = vgg.to(device)
print(summary(model_vgg, (1, 3, 224, 224)))

Make sure to set all parameters of VGG feature extractor not to be trainable

In [ ]:
# MAKE VGG Layers not trainable
# Freeze all the feature extraction layers
for param in model_vgg.features.parameters():
    param.requires_grad = False

# Verify which layers are trainable
trainable_params = sum(p.numel() for p in model_vgg.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_vgg.parameters())
print(f'Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)')

Now we can start the training. Be very patient, as training takes a long time, and our train function is not designed to print anything before the end of the epoch.

In [ ]:
# TRAIN THE MODEL
optimizer_vgg = torch.optim.Adam(model_vgg.classifier.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

history_vgg = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
epochs = 3

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model_vgg, train_loader_vgg, optimizer_vgg, loss_fn)
    val_loss, val_acc = validate(model_vgg, test_loader_vgg, loss_fn)
    
    history_vgg['train_loss'].append(train_loss)
    history_vgg['train_acc'].append(train_acc)
    history_vgg['val_loss'].append(val_loss)
    history_vgg['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1}/{epochs}: '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

# Plot training results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history_vgg['train_acc'], label='Train Accuracy')
ax1.plot(history_vgg['val_acc'], label='Validation Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('VGG Transfer Learning - Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history_vgg['train_loss'], label='Train Loss')
ax2.plot(history_vgg['val_loss'], label='Validation Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('VGG Transfer Learning - Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f'\nFinal Validation Accuracy: {history_vgg["val_acc"][-1]*100:.2f}%')

It seems much better now!

## Optional: Calculate Top 3 Accuracy

We can also computer Top 3 accuracy using the same code as in the previous exercise.


In [ ]:
# CALCULATE TOP-3 Accuracy of the model
def top_k_accuracy(model, dataloader, k=3):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Get top-k predictions
            _, top_k_preds = outputs.topk(k, dim=1)
            
            # Check if true label is in top-k predictions
            correct += top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds)).sum().item()
            total += labels.size(0)
    
    return correct / total

# Calculate Top-1 and Top-3 accuracy
top1_acc = top_k_accuracy(model_vgg, test_loader_vgg, k=1)
top3_acc = top_k_accuracy(model_vgg, test_loader_vgg, k=3)

print(f'Top-1 Accuracy: {top1_acc*100:.2f}%')
print(f'Top-3 Accuracy: {top3_acc*100:.2f}%')

# Compare with simple CNN
print(f'\nComparison:')
print(f'Simple CNN Top-1 Accuracy: {history["val_acc"][-1]*100:.2f}%')
print(f'VGG Transfer Learning Top-1 Accuracy: {top1_acc*100:.2f}%')
print(f'VGG Transfer Learning Top-3 Accuracy: {top3_acc*100:.2f}%')